<a href="https://colab.research.google.com/github/PalakPrajapati346/WORKSHOP-2/blob/main/file2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:




# 2. FEATURE ENGINEERING



# 3. DEFINE CATEGORICAL COLUMNS
# CatBoost handles these automatically if we tell it which ones they are


# 4. CATBOOST WITH 5-FOLD CROSS VALIDATION



# 5. CALCULATE HACKATHON SCORE (Exact Metric from Image)


# 6. EXPORT SUBMISSION (Matches guidelines: 41778 x 2)


In [2]:
!pip install catboost

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 10.5 MB/s eta 0:00:00


In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score
from catboost import CatBoostRegressor

In [4]:
# 1. LOAD DATA
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [20]:
def prepare_features(df):
    df_copy = df.copy()

    # Time logic
    df_copy['timestamp'] = pd.to_datetime(df_copy['timestamp'], format='%H:%M')
    df_copy['hour'] = df_copy['timestamp'].dt.hour
    df_copy['minute'] = df_copy['timestamp'].dt.minute
    df_copy['total_min'] = df_copy['hour'] * 60 + df_copy['minute']
    df_copy['sin_time'] = np.sin(2 * np.pi * df_copy['total_min'] / 1440)
    df_copy['cos_time'] = np.cos(2 * np.pi * df_copy['total_min'] / 1440)

    # Force Numeric columns (Safe from 'missing' strings)
    num_cols = ['NumberofLanes', 'Temperature']
    for col in num_cols:
        df_copy[col] = pd.to_numeric(df_copy[col], errors='coerce')
        df_copy[col] = df_copy[col].fillna(df_copy[col].median())

    # Force Categorical columns
    cat_cols = ['geohash', 'day', 'RoadType', 'Weather', 'LargeVehicles', 'Landmarks']
    for col in cat_cols:
        df_copy[col] = df_copy[col].astype(str).replace('nan', 'missing').replace('None', 'missing')

    # Drop and Return
    final_df = df_copy.drop(columns=['timestamp', 'minute', 'total_min'])
    return final_df

In [22]:
X_train_full = prepare_features(train)
y_train_full = train['demand']
X_test_full = prepare_features(test)

# ALIGNMENT FIX: Only align columns that are in the features
X_test_full = X_test_full[[c for c in X_train_full.columns if c in X_test_full.columns]]
X_train_full = X_train_full[X_test_full.columns]

In [23]:
cat_features_indices = [X_train_full.columns.get_loc(c) for c in ['geohash', 'day', 'RoadType', 'Weather', 'LargeVehicles', 'Landmarks']]

In [24]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X_train_full))
test_preds = np.zeros(len(X_test_full))

In [25]:
for fold, (tr_idx, va_idx) in enumerate(kf.split(X_train_full, y_train_full)):
    X_tr, X_va = X_train_full.iloc[tr_idx], X_train_full.iloc[va_idx]
    y_tr, y_va = y_train_full.iloc[tr_idx], y_train_full.iloc[va_idx]

    model = CatBoostRegressor(
        iterations=1500, # Slightly lower to save time before deadline
        learning_rate=0.05,
        depth=8,
        eval_metric='R2',
        random_seed=42,
        verbose=500
    )

    model.fit(
        X_tr, y_tr,
        cat_features=cat_features_indices,
        eval_set=(X_va, y_va),
        early_stopping_rounds=100
    )

    oof_preds[va_idx] = model.predict(X_va)
    test_preds += model.predict(X_test_full) / 5

0:	learn: 0.0726943	test: 0.0739846	best: 0.0739846 (0)	total: 35.7ms	remaining: 53.6s
500:	learn: 0.9387757	test: 0.9340348	best: 0.9340348 (500)	total: 21s	remaining: 41.8s
1000:	learn: 0.9517936	test: 0.9413498	best: 0.9413498 (1000)	total: 42.7s	remaining: 21.3s
1499:	learn: 0.9588007	test: 0.9451915	best: 0.9451915 (1499)	total: 1m 4s	remaining: 0us

bestTest = 0.9451915389
bestIteration = 1499

0:	learn: 0.0726529	test: 0.0720462	best: 0.0720462 (0)	total: 33ms	remaining: 49.5s
500:	learn: 0.9395831	test: 0.9337331	best: 0.9337331 (500)	total: 21.6s	remaining: 43.1s
1000:	learn: 0.9523972	test: 0.9423118	best: 0.9423239 (999)	total: 43.8s	remaining: 21.8s
1499:	learn: 0.9585933	test: 0.9454711	best: 0.9454711 (1499)	total: 1m 6s	remaining: 0us

bestTest = 0.9454711408
bestIteration = 1499

0:	learn: 0.0725917	test: 0.0714090	best: 0.0714090 (0)	total: 50.2ms	remaining: 1m 15s
500:	learn: 0.9416232	test: 0.9368185	best: 0.9368185 (500)	total: 20.4s	remaining: 40.8s
1000:	learn: 0.

In [26]:
final_r2 = r2_score(y_train_full, oof_preds)
print(f"\nFINAL PROJECTED HACKATHON SCORE: {max(0, 100 * final_r2):.4f}")


FINAL PROJECTED HACKATHON SCORE: 94.5199


In [27]:
submission = pd.DataFrame({'Index': test['Index'], 'demand': test_preds})
submission['demand'] = submission['demand'].clip(lower=0)
submission.to_csv('submission.csv', index=False)